# 💼 Predicting Your Future with Census Data
Este notebook implementa un sistema de recomendación basado en datos del censo estadounidense (Adult Census Income Dataset).
El objetivo es estimar la probabilidad de que una persona gane más de 50K anuales a los 40 años, en base a su perfil sociodemográfico.

**Modelo utilizado:** K-Nearest Neighbors (KNN)

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

In [9]:
# Cargar y limpiar los datos
df = pd.read_csv("adult-census-income.csv")
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)
df['income'] = df['income'].str.strip()
df['high_income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

In [10]:
# Selección de variables y preprocesamiento
features = ['age', 'education', 'marital.status', 'occupation', 'hours.per.week', 'sex', 'native.country']
X = df[features]
y = df['high_income']

numeric_features = ['age', 'hours.per.week']
categorical_features = ['education', 'marital.status', 'occupation', 'sex', 'native.country']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [11]:
# Buscar el mejor K usando validación cruzada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scores = []
k_range = range(19, 41)

for k in k_range:
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=k))
    ])
    cv_score = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    scores.append(cv_score.mean())
    print(f"K={k:2d} → Accuracy CV: {cv_score.mean()*100:.1f}%")

best_k = k_range[scores.index(max(scores))]
print(f"\nMejor K: {best_k} con accuracy {max(scores)*100:.1f}%")

K=19 → Accuracy CV: 82.4%
K=20 → Accuracy CV: 82.5%
K=21 → Accuracy CV: 82.4%
K=22 → Accuracy CV: 82.4%
K=23 → Accuracy CV: 82.4%
K=24 → Accuracy CV: 82.5%
K=25 → Accuracy CV: 82.4%
K=26 → Accuracy CV: 82.6%
K=27 → Accuracy CV: 82.4%
K=28 → Accuracy CV: 82.6%
K=29 → Accuracy CV: 82.4%
K=30 → Accuracy CV: 82.5%
K=31 → Accuracy CV: 82.5%
K=32 → Accuracy CV: 82.5%
K=33 → Accuracy CV: 82.5%
K=34 → Accuracy CV: 82.5%
K=35 → Accuracy CV: 82.5%
K=36 → Accuracy CV: 82.5%
K=37 → Accuracy CV: 82.5%
K=38 → Accuracy CV: 82.6%
K=39 → Accuracy CV: 82.6%
K=40 → Accuracy CV: 82.7%

Mejor K: 40 con accuracy 82.7%


In [12]:
# Entrenar el modelo final con el mejor K
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=best_k))
])
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Accuracy final en test con K={best_k}: {accuracy*100:.1f}%")

Accuracy final en test con K=40: 82.4%


In [13]:
# Función de recomendación
def recommend_trajectory(user_profile):
    """
    user_profile debe ser un diccionario con keys:
    - age, education, marital.status, occupation, hours.per.week, sex, native.country
    """
    user_df = pd.DataFrame([user_profile])
    prediction = model.predict(user_df)[0]
    prob = model.predict_proba(user_df)[0][1]
    if prediction == 1:
        return f"✅ Con este perfil, tu probabilidad de ganar más de 50K es del {prob*100:.1f}%."
    else:
        return f"⚠️ Con este perfil, tu probabilidad de superar los 50K es solo del {prob*100:.1f}%. Considera mejorar tu educación o cambiar de ocupación."

In [14]:
# Simulación de usuario
simulated_user = {
    "age": 25,
    "education": "HS-grad",
    "marital.status": "Never-married",
    "occupation": "Sales",
    "hours.per.week": 30,
    "sex": "Male",
    "native.country": "United-States"
}
recommend_trajectory(simulated_user)

'⚠️ Con este perfil, tu probabilidad de superar los 50K es solo del 0.0%. Considera mejorar tu educación o cambiar de ocupación.'